![Week 2 Day 1](../../assets/banners/w2d1.png)

# Week 2 Day 1 - SOLUTION - Growing the Toy Embedding

Reference solution for the exercise at the end of `week2/day1.ipynb`. Have a go yourself first: struggling a little is where the learning happens! Then compare.

**The exercise:** add *mutex*, *subnet* and *index* to the 3-D toy world and predict their neighbours; find a case where Euclidean distance and cosine similarity disagree; bonus, plot real embeddings of your own sentences.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def cosine(a, b) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


# [operating systems, networks, databases]
toy_3d = {
    "process": [0.90, 0.10, 0.05],
    "deadlock": [0.80, 0.05, 0.30],
    "packet": [0.05, 0.95, 0.00],
    "TCP": [0.15, 0.85, 0.05],
    "SQL": [0.05, 0.05, 0.95],
    "transaction": [0.25, 0.05, 0.90],
    "lock": [0.65, 0.05, 0.60],
    # ✅ the new words
    "mutex": [0.90, 0.02, 0.20],  # an OS lock, with a little database flavour
    "subnet": [0.02, 0.95, 0.02],  # pure networking
    "index": [0.10, 0.05, 0.85],  # B+ tree indexes are a database idea
}

## Part 1: Predict, then check

My predictions before computing: *mutex* → *process* or *lock*; *subnet* → *packet*; *index* → *SQL*.

In [ ]:
for word in ("mutex", "subnet", "index"):
    others = [w for w in toy_3d if w != word]
    best = max(others, key=lambda w: cosine(toy_3d[word], toy_3d[w]))
    print(f"{word:<7} nearest by cosine → {best:<12} ({cosine(toy_3d[word], toy_3d[best]):.3f})")

## Part 2: When Euclidean distance and cosine disagree

They disagree when vectors have different **lengths**. Take a short "deadlock" vector and two candidates: a long vector pointing exactly the same way (a whole page about deadlocks) and a short vector pointing slightly differently.

In [ ]:
query = np.array(toy_3d["deadlock"])
candidates = {
    "long page about deadlocks": query * 4,  # same direction, 4× longer
    "short note on mutexes": np.array(toy_3d["mutex"]),  # different direction, similar length
}
for name, vec in candidates.items():
    print(f"{name:<27} cosine = {cosine(query, vec):.3f}   euclidean = {np.linalg.norm(query - vec):.3f}")

by_cos = max(candidates, key=lambda n: cosine(query, candidates[n]))
by_euc = min(candidates, key=lambda n: np.linalg.norm(query - candidates[n]))
print(f"\nNearest by cosine   : {by_cos}\nNearest by Euclidean: {by_euc}  ← they disagree")

Cosine says the long page is a perfect match (same direction); Euclidean says the mutex note is closer (similar length). For text, "about the same thing" is what we want, which is why we use cosine. With unit-length vectors (as OpenAI's embeddings are), the two always agree.

## Part 3 (bonus): your own sentences, with real embeddings

This needs an API key, so we catch the friendly error if there isn't one.

In [ ]:
from sklearn.decomposition import PCA

from gradewise.errors import MissingAPIKeyError
from gradewise.models import embeddings

my_sentences = [
    "A mutex lets only one thread into the critical section.",
    "A semaphore counts how many threads may enter.",
    "A subnet mask splits an IP address into network and host parts.",
    "Routers forward packets using the network part of the address.",
    "An index makes lookups fast but slows down every insert.",
    "A B+ tree keeps keys sorted so range queries are quick.",
]
try:
    vecs = np.array(embeddings().embed_documents(my_sentences))
    points = PCA(n_components=2).fit_transform(vecs)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(points[:, 0], points[:, 1], color="#7c5cff")
    for (x, y), text in zip(points, my_sentences, strict=True):
        ax.text(x, y, " " + text[:40] + "…", fontsize=9)
    plt.show()
except MissingAPIKeyError as err:
    err.show()
    print("No key, no problem: with one you'd see three tidy pairs (OS, networks, databases).")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Real vectors always come from <code>gradewise.models.embeddings()</code>. The toy ones are for intuition only.</span>
        </td>
    </tr>
</table>